In [1]:
import cv2
import mediapipe as mp
import csv
import sys

DATA_FILE = "gestures.csv"
GESTURES = ["Three_fingers"]
current_idx = 0

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.7)
cap = cv2.VideoCapture(0)
count = 0

while cap.isOpened():
    success, frame = cap.read()
    if not success: break
    
    label = GESTURES[current_idx]
    frame = cv2.flip(frame, 1)
    results = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

    if results.multi_hand_landmarks:
        lm = results.multi_hand_landmarks[0]
        mp.solutions.drawing_utils.draw_landmarks(frame, lm, mp_hands.HAND_CONNECTIONS)
        
        key = cv2.waitKey(1)
        if key == ord(' '):
            row = [label] + [val for p in lm.landmark for val in (p.x, p.y, p.z)]
            with open(DATA_FILE, 'a', newline='') as f:
                csv.writer(f).writerow(row)
            count += 1
        
        # NEXT GESTURE
        elif key == ord('n'):
            current_idx += 1
            if current_idx >= len(GESTURES):
                print("All gestures recorded. Exiting...")
                break
            count = 0 # Reset counter for next gesture
            print(f"Switching to: {GESTURES[current_idx]}")

        # EXIT SCRIPT
        elif key == 27: # ESC Key
            print("Saving and exiting...")
            break

    # UI Overlay
    cv2.rectangle(frame, (0,0), (350, 60), (0,0,0), -1)
    cv2.putText(frame, f"REC: {label.upper()}", (10, 25), 1, 1.5, (0, 255, 0), 2)
    cv2.putText(frame, f"COUNT: {count}", (10, 50), 1, 1.2, (255, 255, 255), 1)
    
    cv2.imshow('Society Data Collector', frame)
    if cv2.waitKey(1) == 27: break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1770483894.294084       1 gl_context.cc:344] GL version: 2.1 (2.1 Metal - 89.4), renderer: Apple M4
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


: 